# Hugging Face pipeline and LangChain wrapper

This Colab-compatible notebook compares direct `model.generate()`, `transformers.pipeline()`, and LangChain `HuggingFacePipeline`. The tiny model demonstrates APIs rather than answer quality.

In [ ]:
%pip install -q "transformers==4.57.6" "torch>=2.7,<3" "langchain-huggingface==1.2.2" "langchain-core==1.5.1"

In [ ]:
from time import perf_counter

from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_name = "sshleifer/tiny-gpt2"
prompt = "Enterprise evidence should be cited because"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
try:
    started = perf_counter()
    inputs = tokenizer(prompt, return_tensors="pt")
    tokens = model.generate(**inputs, max_new_tokens=20, do_sample=False)
    direct = tokenizer.decode(tokens[0], skip_special_tokens=True)
    direct_ms = (perf_counter() - started) * 1000

    generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
    started = perf_counter()
    piped = generator(prompt, max_new_tokens=20, do_sample=False)[0]["generated_text"]
    pipeline_ms = (perf_counter() - started) * 1000

    langchain_llm = HuggingFacePipeline(
        pipeline=generator,
        pipeline_kwargs={
            "max_new_tokens": 20,
            "do_sample": False,
            "return_full_text": False,
        },
    )
    started = perf_counter()
    wrapped = langchain_llm.invoke(prompt)
    langchain_ms = (perf_counter() - started) * 1000
    print(
        {
            "direct_ms": direct_ms,
            "pipeline_ms": pipeline_ms,
            "langchain_ms": langchain_ms,
        }
    )
    print({"direct": direct, "pipeline": piped, "langchain": wrapped})
except Exception as exc:  # noqa: BLE001 - demo reports runtime/model compatibility
    print(f"Demo could not run in this runtime: {type(exc).__name__}: {exc}")

In [ ]:
try:
    summarizer = pipeline("summarization", model="hf-internal-testing/tiny-random-t5")
    print(
        summarizer(
            "EnterpriseRAG retrieves evidence and validates citations before returning an answer.",
            max_new_tokens=20,
        )
    )
except Exception as exc:  # noqa: BLE001 - optional tiny model may be unavailable
    print(f"Tiny summarization pipeline unavailable: {type(exc).__name__}: {exc}")

## Expected output

The timing dictionary contains three non-negative measurements and each API returns text. Tiny random models may produce low-quality text; this notebook validates integration mechanics.